# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available RecordSets, Fields, and their `@id`s.

In [ ]:
# List all record sets in the dataset using their @id
record_sets = list(dataset.record_sets)

if not record_sets:
    print('No RecordSets were found in the metadata.')
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Description: {rs.description if hasattr(rs, 'description') else 'No description'}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
        print()

## 3. Data Extraction
Load data from each available RecordSet into a DataFrame for analysis. Use the RecordSet and field `@id`s as shown above.

In [ ]:
# Identify available RecordSet @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records found.")

if not dataframes:
    print('No tabular data was extracted. Please check the record set definitions or contact the data provider.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

> **Note:** Replace the values of `selected_record_set_id`, `numeric_field_id`, and `group_field_id` below with valid `@id`s obtained from the Data Overview.

In [ ]:
# Example setup: replace these @ids with actual ones from your dataset if available
# If the overview above printed out @ids for a tabular record set, use one of them here
if dataframes:
    # Example: select the first record set
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]

    # Attempt to identify a numeric field (column)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id:
        print(f"Using numeric field '@id': {numeric_field_id}\n")
        threshold = df[numeric_field_id].mean()  # Example threshold: mean value
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to pick a group field (categorical column)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id} (first 5 rows):")
            display(grouped_df.head())
        else:
            print("No categorical/grouping field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print('No record sets loaded with tabular data. Skipping EDA.')

## 5. Visualization
Visualize numeric field distributions or relationships. This example shows a histogram and, if possible, a boxplot by group.

> **Note:** If there are no numeric or categorical fields, this section will not render plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No suitable numeric or grouping field for visualization.')

## 6. Conclusion
This notebook demonstrated how to access and explore a FAIR² dataset using the `mlcroissant` library. We loaded metadata, listed RecordSets and Fields by their `@id`, extracted available data, and performed basic exploration, normalization, and visualization. For full utilization, review the schema's details and replace field identifiers as needed.